# Detecting Re-injury Risk in NBA Players with an EWMA-CUSUM Control Chart

This notebook contains the full analysis pipeline for monitoring NBA players' on-court performance after they return from injury, using a lower-sided EWMA-CUSUM control chart to flag performance declines that may precede a second injury.

**Data.** Regular-season player and team box scores for the 2020-21 through 2024-25 seasons (hoopR `seasons = 2021:2025`, where a season is labelled by the calendar year in which it ends), loaded with the [`hoopR`](https://hoopr.sportsdataverse.org/) package. Injury absences are identified from the `did_not_play` / `reason` fields of the player box scores.

**Performance metrics monitored.** PIE, Usage Rate, a simplified box plus-minus (BPM), and Net Rating (Dean Oliver offensive/defensive ratings).

**Pipeline overview**

| Part | Steps | Contents |
|---|---|---|
| 1 | 1–14 | Data preparation: load box scores, identify qualifying injury stints, return-to-play and subsequent re-injury, and compute per-game metrics |
| 2 | 15–19 | Control-chart machinery: EWMA-CUSUM statistic, parameter grid, player-specific empirical-Bayes (shrinkage) baselines, chart runners and confusion-matrix summaries |
| 3 | 20–26 | Evaluation: baselines fit on training seasons only, block-bootstrap check for autocorrelation, (λ, h) selected on training seasons, and a single application of the frozen model to the held-out season |
| 4 | 27–28 | ROC/AUC comparison of the league-average and individualized baselines across a wide (λ, h) grid (training seasons) |
| 5 | 29–31 | Figures: parameter-grid heatmaps (overall and by injury area) and an example control chart |
| 6 | 32–41 | Descriptive and supplementary statistics |

Two baseline approaches are compared throughout:

* **Approach A — league average:** the in-control mean and SD are the prior season's league-wide mean and SD for the metric.
* **Approach B — individualized baseline:** a credibility-weighted blend of the player's own pre-injury mean and the league grand mean (empirical-Bayes shrinkage, with the shrinkage weight *k* estimated by one-way random-effects ANOVA method of moments).

**Running the notebook.** Run all cells top to bottom with an R kernel (e.g. Google Colab with the R runtime, or Jupyter with IRkernel). The first run downloads several seasons of box scores and the bootstrap/grid steps take a while. Figures are written as PNG files to the working directory.

## Setup

In [ ]:
required_pkgs <- c("tidyverse", "hoopR", "patchwork")
missing_pkgs  <- setdiff(required_pkgs, rownames(installed.packages()))
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

In [ ]:
library(tidyverse)
library(hoopR)
library(patchwork)

# join_by() inequality joins, pick() and reframe() require dplyr >= 1.1.0
stopifnot(packageVersion("dplyr") >= "1.1.0")

### Parameters and lookup tables

In [ ]:
seasons <- 2021:2025      # hoopR season = calendar year in which the season ends

bootstrap_seed <- 2026    # seed for the block bootstrap (Step 23)

non_inj_reasons <- c(
  "ASTHMA ATTACK", "COACH'S DECISION", "DENTAL PAIN",
  "DID NOT DRESS", "FAMILY MATTERS", "FLU", "FLU LIKE SYMPTOMS",
  "FOOD POISONING", "GASTROENTERITIS", "HEALTH & SAFETY PROTOCOLS",
  "ILLNESS", "MIGRAINE", "MONONUCLEOSIS", "NON-COVID ILLNESS",
  "ORAL SURGERY", "RESPIRATORY", "SICK", "SINUSES", "SORE THROAT",
  "STOMACH", "STOMACH ILLNESS", "STOMACH VIRUS",
  "UPPER RESPIRATORY INFECTION", "VIRUS", "SUSPENDED BY LEAGUE",
  "TRADED",
  "G-LEAGUE ASSIGNMENT", "NOT WITH TEAM",
  "NOT WITH TEAM - G LEAGUE ASSIGNMENT", "PERSONAL",
  "PERSONAL REASONS", "REST", "RETURN TO COMPETITION CONDITIONING",
  "TRADE PENDING", "TWO-WAY CONTRACT",
  "RETURN TO COMPETITION RECONDITIONING",
  "SURGERY REHAB",
  "SUSPENDED BY TEAM"
)

lower_extremity_keywords <- c(
  "ACHILLES", "ANKLE", "CALF", "FOOT", "HAMSTRING", "HIP",
  "KNEE", "LEG", "PATELLAR", "PLANTAR", "QUAD", "QUADRICEP",
  "SHIN", "THIGH", "TOE", "GROIN", "GLUTE", "GLUTEAL",
  "ACL", "MCL", "PCL", "MENISCUS", "ADDUCTOR", "ABDUCTOR",
  "SOLEUS", "TIBIAL", "TIBIA", "FIBULA", "HEEL", "PELVIC",
  "PELVIS", "TAILBONE", "PATELLA", "GREAT TOE"
)

upper_extremity_keywords <- c(
  "ELBOW", "FINGER", "FOREARM", "HAND", "SHOULDER", "THUMB",
  "WRIST", "BICEP", "TRICEP", "ROTATOR CUFF", "COLLARBONE",
  "CLAVICLE", "AC JOINT", "LABRUM", "ARM", "UCL", "ULNAR"
)

core_keywords <- c(
  "ABDOMEN", "ABDOMINAL", "BACK", "CORE", "HERNIA", "LUMBAR",
  "NECK", "RIB", "RIBS", "SPINE", "THORACIC", "OBLIQUE",
  "STERNUM", "CHEST", "CERVICAL", "WHIPLASH"
)

head_keywords <- c(
  "HEAD", "CONCUSSION", "FACE", "FACIAL", "NOSE", "NASAL",
  "EYE", "SINUS"
)

clean_reason <- function(reason) {
  reason |>
    str_to_upper() |>
    str_trim() |>
    str_remove("^(LEFT|RIGHT|BILATERAL)\\s+") |>
    str_remove(paste0(
      "^(SORE|STRAINED|SPRAINED|TORN|TIGHT|TIGHTNESS|INFLAMED|",
      "BRUISED|BROKEN|FRACTURED|DISLOCATED|SWOLLEN|SURGERY|",
      "SORENESS|TENDINITIS|STRESS FRACTURE|BONE BRUISE)\\s+"
    ))
}

# Body-region classification; checked in order (first match wins)
classify_injury <- function(reason_cleaned) {
  case_when(
    str_detect(reason_cleaned, paste(head_keywords,            collapse = "|")) ~ "Head",
    str_detect(reason_cleaned, paste(core_keywords,            collapse = "|")) ~ "Core",
    str_detect(reason_cleaned, paste(upper_extremity_keywords, collapse = "|")) ~ "Upper Extremity",
    str_detect(reason_cleaned, paste(lower_extremity_keywords, collapse = "|")) ~ "Lower Extremity",
    TRUE ~ "Other"
  )
}

## Part 1 — Data preparation

### Step 1. Load box scores (regular season only)

In [ ]:
team_box_raw <- load_nba_team_box(seasons = seasons) |>
  filter(season_type == 2) |>
  select(
    game_id, game_date, team_display_name, team_score,
    field_goals_made, field_goals_attempted,
    three_point_field_goals_made,
    free_throws_made, free_throws_attempted,
    offensive_rebounds, defensive_rebounds, total_rebounds,
    assists, steals, blocks, turnovers, fouls
  ) |>
  rename(Team = team_display_name) |>
  mutate(
    game_date = as.Date(game_date),
    Season    = lubridate::year(game_date) +
                if_else(lubridate::month(game_date) >= 10, 1, 0)
  )

player_box_raw <- load_nba_player_box(seasons = seasons) |>
  filter(season_type == 2) |>
  mutate(
    game_date = as.Date(game_date),
    Team      = paste(team_location, team_name)
  ) |>
  rename(Season = season)

### Step 2. Game number within each team-season

In [ ]:
team_box <- team_box_raw |>
  group_by(Team, Season) |>
  arrange(game_date, .by_group = TRUE) |>
  mutate(game_number = row_number()) |>
  ungroup() |>
  filter(game_number <= 82)

### Step 3. Team minutes
Computed from the raw player box (before any player filtering) so team totals are complete.

In [ ]:
team_minutes <- player_box_raw |>
  filter(did_not_play == FALSE, active == TRUE, minutes > 0) |>
  mutate(minutes = as.numeric(minutes)) |>
  group_by(game_id, Team) |>
  summarise(t_minutes = sum(minutes, na.rm = TRUE), .groups = "drop")

### Step 4. Team game log
Team box-score stats joined with opponent stats and team/opponent minutes.

In [ ]:
# Self-join on game_id pairs each team with both teams in the game
# (hence many-to-many); the Team != Opp_Team filter keeps the opponent row.
opponent_stats <- team_box |>
  inner_join(
    team_box |>
      select(
        game_id,
        Opp_Team = Team,
        Opp_PTS  = team_score,
        Opp_FGM  = field_goals_made,
        Opp_FGA  = field_goals_attempted,
        Opp_FTM  = free_throws_made,
        Opp_FTA  = free_throws_attempted,
        Opp_ORB  = offensive_rebounds,
        Opp_DRB  = defensive_rebounds,
        Opp_TRB  = total_rebounds,
        Opp_AST  = assists,
        Opp_STL  = steals,
        Opp_BLK  = blocks,
        Opp_TOV  = turnovers,
        Opp_PF   = fouls
      ),
    by = "game_id",
    relationship = "many-to-many"
  ) |>
  filter(Team != Opp_Team) |>
  select(game_id, Team, starts_with("Opp_"))

opponent_minutes <- team_minutes |>
  rename(Opp_Team = Team, Opp_t_minutes = t_minutes) |>
  left_join(
    team_box |> select(game_id, Team) |>
      inner_join(team_box |> select(game_id, Opp_Team = Team),
                 by = "game_id", relationship = "many-to-many") |>
      filter(Team != Opp_Team),
    by = c("game_id", "Opp_Team")
  ) |>
  select(game_id, Team, Opp_t_minutes) |>
  distinct()

team_gamelog <- team_box |>
  rename(
    t_pts  = team_score,
    t_fgm  = field_goals_made,
    t_fga  = field_goals_attempted,
    t_3pm  = three_point_field_goals_made,
    t_ftm  = free_throws_made,
    t_fta  = free_throws_attempted,
    t_oreb = offensive_rebounds,
    t_dreb = defensive_rebounds,
    t_treb = total_rebounds,
    t_ast  = assists,
    t_stl  = steals,
    t_blk  = blocks,
    t_to   = turnovers,
    t_pf   = fouls
  ) |>
  left_join(team_minutes,     by = c("game_id", "Team")) |>
  left_join(opponent_stats,   by = c("game_id", "Team")) |>
  left_join(opponent_minutes, by = c("game_id", "Team")) |>
  select(-game_number)

### Step 5. Restrict to players with a single team per season, and attach game number

In [ ]:
single_team_players <- player_box_raw |>
  distinct(athlete_id, Season, Team) |>
  group_by(athlete_id, Season) |>
  filter(n() == 1) |>
  ungroup()

player_box <- player_box_raw |>
  semi_join(single_team_players, by = c("athlete_id", "Season"))

player_full <- player_box |>
  left_join(
    team_box |> select(game_id, Team, Season, game_number),
    by = c("game_id", "Team", "Season")
  )

### Step 6. Injury-related DNPs and injury classification

In [ ]:
injury_data <- player_full |>
  filter(
    did_not_play == TRUE,
    !(reason %in% non_inj_reasons)
  ) |>
  mutate(
    reason_upper   = str_to_upper(str_trim(reason)),
    reason_cleaned = clean_reason(reason_upper),
    injury_type    = classify_injury(reason_cleaned)
  ) |>
  arrange(athlete_id, game_date)

### Step 7. Injury stints
Consecutive injury DNPs separated by no more than 5 team games are grouped into one stint.

In [ ]:
get_mode <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA_character_)
  names(sort(table(x), decreasing = TRUE))[1]
}

injury_stints <- injury_data |>
  group_by(athlete_id, Season) |>
  arrange(game_number, .by_group = TRUE) |>
  mutate(
    missed_gap   = game_number - lag(game_number),
    new_stint    = if_else(is.na(missed_gap) | missed_gap > 5, 1, 0),
    Injury_Stint = cumsum(new_stint)
  ) |>
  ungroup() |>
  group_by(athlete_id, Season, Injury_Stint) |>
  summarise(
    first_game_missed     = min(game_number),
    last_game_missed      = max(game_number),
    games_missed_in_stint = n(),
    stint_span            = last_game_missed - first_game_missed + 1,
    injury_type           = get_mode(injury_type),
    primary_reason        = get_mode(reason_upper),
    .groups = "drop"
  )

### Step 8. Return game for each stint

In [ ]:
played_games <- player_full |>
  filter(did_not_play == FALSE, active == TRUE, minutes > 0) |>
  select(athlete_id, Season, game_number)

injury_stints_with_return <- injury_stints |>
  left_join(played_games, by = c("athlete_id", "Season")) |>
  filter(game_number > last_game_missed) |>
  group_by(athlete_id, Season, Injury_Stint) |>
  filter(game_number == min(game_number)) |>
  rename(game_returned = game_number) |>
  ungroup() |>
  mutate(
    actual_games_missed = game_returned - first_game_missed,
    games_until_return  = game_returned - last_game_missed
  )

### Step 9. Qualifying-stint filters

In [ ]:
qualifying_stints <- injury_stints_with_return |>
  filter(
    stint_span         >= 2,
    games_until_return <= 4
  )

### Step 10. First qualifying stint per player-season and the next injury

The subsequent-injury lookup uses an inequality join (`join_by(..., Injury_Stint < next_stint)`). A plain equi-join on `athlete_id` + `Season` would match each stint to itself, and players with no later injury would then be dropped entirely rather than retained with `had_second_injury = FALSE`. With the inequality join, a player with no later stint keeps a single row with `next_stint = NA`.

In [ ]:
first_stints <- qualifying_stints |>
  group_by(athlete_id, Season) |>
  slice_min(Injury_Stint, n = 1) |>
  ungroup()

all_stints_lookup <- injury_stints_with_return |>
  select(athlete_id, Season, Injury_Stint, first_game_missed)

qualifying_with_next_injury <- first_stints |>
  left_join(
    all_stints_lookup |>
      rename(next_stint             = Injury_Stint,
             next_first_game_missed = first_game_missed),
    by = join_by(athlete_id, Season, Injury_Stint < next_stint)
  ) |>
  group_by(athlete_id, Season, Injury_Stint) |>
  arrange(next_stint, .by_group = TRUE) |>   # NA sorts last; real matches ascending
  slice(1) |>                                # soonest subsequent stint, or the lone NA row
  ungroup() |>
  rename(
    second_injury_stint      = next_stint,
    second_injury_first_game = next_first_game_missed
  ) |>
  mutate(
    had_second_injury   = !is.na(second_injury_stint),
    analysis_window_end = if_else(had_second_injury,
                                  second_injury_first_game - 1L, 82L)
  )

# Class balance check: both TRUE and FALSE should be present
qualifying_with_next_injury |> count(had_second_injury)

### Step 11. Helper functions for per-game metrics

* `compute_box_metrics()` adds PIE, Usage Rate and BPM. It expects the player box columns, the matching `team_gamelog` columns, and `lg_ts` (league true-shooting % for the season, from `league_ts_by_season`, Step 12).
* `compute_oliver_ratings()` adds ORtg, DRtg and NetRtg (Dean Oliver / Basketball-Reference formulas). It expects the player box columns, the `team_gamelog` columns (`t_*`, `Opp_*`, `t_minutes`, `Opp_t_minutes`) and `minutes`.

Both are shared by the league reference table (Step 12), the post-return window (Steps 13–14) and the pre-injury panel (Step 17), so every part of the pipeline uses identical metric definitions.

In [ ]:
compute_box_metrics <- function(df) {
  df |>
    mutate(
      p_pie_num = points + field_goals_made + free_throws_made -
                  field_goals_attempted - free_throws_attempted +
                  defensive_rebounds + 0.5 * offensive_rebounds +
                  assists + steals + 0.5 * blocks - fouls - turnovers,
      t_pie_den = t_pts + t_fgm + t_ftm - t_fga - t_fta +
                  t_dreb + 0.5 * t_oreb + t_ast + t_stl + 0.5 * t_blk - t_pf - t_to,
      PIE       = p_pie_num / t_pie_den,

      Usage_Rate = (field_goals_attempted + 0.44 * free_throws_attempted + turnovers) /
                   (t_fga + 0.44 * t_fta + t_to),

      ts_pct = if_else(
        field_goals_attempted + free_throws_attempted == 0, lg_ts,
        points / (2 * (field_goals_attempted + 0.44 * free_throws_attempted))
      ),
      ast_rate  = if_else(minutes == 0, 0, assists / minutes),
      oreb_rate = if_else(minutes == 0, 0, offensive_rebounds / minutes),
      dreb_rate = if_else(minutes == 0, 0, defensive_rebounds / minutes),
      BPM = (ts_pct - lg_ts) * 40 +
            ast_rate * 3 -
            oreb_rate * 1.5 +
            dreb_rate * 2 -
            (turnovers / pmax(minutes, 1)) * 3
    )
}

compute_oliver_ratings <- function(df) {
  df |>
    mutate(
      Team_Scoring_Poss = t_fgm +
        (1 - (1 - if_else(t_fta == 0, 0, t_ftm / t_fta))^2) * t_fta * 0.4,

      Team_Play_pct = if_else(
        (t_fga + t_fta * 0.4 + t_to) == 0, 0,
        Team_Scoring_Poss / (t_fga + t_fta * 0.4 + t_to)
      ),

      Team_ORB_pct = if_else(
        (t_oreb + (Opp_TRB - Opp_ORB)) == 0, 0,
        t_oreb / (t_oreb + (Opp_TRB - Opp_ORB))
      ),

      Team_ORB_Weight = if_else(
        ((1 - Team_ORB_pct) * Team_Play_pct + Team_ORB_pct * (1 - Team_Play_pct)) == 0, 0,
        ((1 - Team_ORB_pct) * Team_Play_pct) /
          ((1 - Team_ORB_pct) * Team_Play_pct + Team_ORB_pct * (1 - Team_Play_pct))
      ),

      qAST = if_else(
        t_minutes == 0 | (t_fgm - field_goals_made) == 0, 0,
        ((minutes / (t_minutes / 5)) *
           (1.14 * ((t_ast - assists) / t_fgm))) +
          ((((t_ast / t_minutes) * minutes * 5 - assists) /
              ((t_fgm / t_minutes) * minutes * 5 - field_goals_made)) *
             (1 - (minutes / (t_minutes / 5))))
      ),

      FG_Part = if_else(
        field_goals_attempted == 0, 0,
        field_goals_made *
          (1 - 0.5 * ((points - free_throws_made) /
                        (2 * field_goals_attempted)) * qAST)
      ),

      AST_Part = if_else(
        (t_fga - field_goals_attempted) == 0, 0,
        0.5 * (((t_pts - t_ftm) - (points - free_throws_made)) /
                 (2 * (t_fga - field_goals_attempted))) * assists
      ),

      FT_Part = if_else(
        free_throws_attempted == 0, 0,
        (1 - (1 - free_throws_made / free_throws_attempted)^2) *
          0.4 * free_throws_attempted
      ),

      ORB_Part = offensive_rebounds * Team_ORB_Weight * Team_Play_pct,

      ScPoss = (FG_Part + AST_Part + FT_Part) *
        (1 - (t_oreb / Team_Scoring_Poss) * Team_ORB_Weight * Team_Play_pct) +
        ORB_Part,

      FGxPoss = if_else(
        field_goals_attempted == 0, 0,
        (field_goals_attempted - field_goals_made) * (1 - 1.07 * Team_ORB_pct)
      ),

      FTxPoss = if_else(
        free_throws_attempted == 0, 0,
        ((1 - free_throws_made / free_throws_attempted)^2) * 0.4 * free_throws_attempted
      ),

      TotPoss = ScPoss + FGxPoss + FTxPoss + turnovers,

      PProd_FG_Part = if_else(
        field_goals_attempted == 0, 0,
        2 * (field_goals_made + 0.5 * three_point_field_goals_made) *
          (1 - 0.5 * ((points - free_throws_made) /
                        (2 * field_goals_attempted)) * qAST)
      ),

      PProd_AST_Part = if_else(
        (t_fgm - field_goals_made) == 0, 0,
        2 * ((t_fgm - field_goals_made + 0.5 * (t_3pm - three_point_field_goals_made)) /
               (t_fgm - field_goals_made)) *
          0.5 * (((t_pts - t_ftm) - (points - free_throws_made)) /
                   (2 * (t_fga - field_goals_attempted))) * assists
      ),

      PProd_ORB_Part = if_else(
        (t_fgm + (1 - (1 - if_else(t_fta == 0, 0, t_ftm / t_fta))^2) *
           0.4 * t_fta) == 0, 0,
        offensive_rebounds * Team_ORB_Weight * Team_Play_pct *
          (t_pts / (t_fgm +
                      (1 - (1 - if_else(t_fta == 0, 0, t_ftm / t_fta))^2) *
                      0.4 * t_fta))
      ),

      PProd = (PProd_FG_Part + PProd_AST_Part + free_throws_made) *
        (1 - (t_oreb / Team_Scoring_Poss) * Team_ORB_Weight * Team_Play_pct) +
        PProd_ORB_Part,

      ORtg = if_else(TotPoss == 0, NA_real_, 100 * PProd / TotPoss),

      DOR_pct = if_else((Opp_ORB + t_dreb) == 0, 0, Opp_ORB / (Opp_ORB + t_dreb)),
      DFG_pct = if_else(Opp_FGA == 0, 0, Opp_FGM / Opp_FGA),

      FMwt = if_else(
        (DFG_pct * (1 - DOR_pct) + (1 - DFG_pct) * DOR_pct) == 0, 0,
        (DFG_pct * (1 - DOR_pct)) /
          (DFG_pct * (1 - DOR_pct) + (1 - DFG_pct) * DOR_pct)
      ),

      Stops1 = steals + blocks * FMwt * (1 - 1.07 * DOR_pct) + defensive_rebounds * (1 - FMwt),

      Stops2 = if_else(
        t_minutes == 0 | t_pf == 0, 0,
        (((Opp_FGA - Opp_FGM - t_blk) / t_minutes) * FMwt * (1 - 1.07 * DOR_pct) +
           ((Opp_TOV - t_stl) / t_minutes)) * minutes +
          (fouls / t_pf) * 0.4 * Opp_FTA *
          (1 - if_else(Opp_FTA == 0, 0, Opp_FTM / Opp_FTA))^2
      ),

      Stops = Stops1 + Stops2,

      Team_Poss = 0.5 * (
        (t_fga + 0.4 * t_fta - 1.07 * Team_ORB_pct * (t_fga - t_fgm) + t_to) +
          (Opp_FGA + 0.4 * Opp_FTA - 1.07 * DOR_pct * (Opp_FGA - Opp_FGM) + Opp_TOV)
      ),

      Stop_pct = if_else(
        Team_Poss == 0 | minutes == 0, 0,
        (Stops * coalesce(Opp_t_minutes, 240)) / (Team_Poss * minutes)
      ),

      Team_DRtg = if_else(Team_Poss == 0, NA_real_, 100 * Opp_PTS / Team_Poss),

      D_Pts_per_ScPoss = if_else(
        (Opp_FGM + (1 - (1 - if_else(Opp_FTA == 0, 0, Opp_FTM / Opp_FTA))^2) *
           0.4 * Opp_FTA) == 0, NA_real_,
        Opp_PTS / (Opp_FGM +
                     (1 - (1 - if_else(Opp_FTA == 0, 0, Opp_FTM / Opp_FTA))^2) *
                     0.4 * Opp_FTA)
      ),

      DRtg = if_else(
        is.na(Team_DRtg) | is.na(D_Pts_per_ScPoss), NA_real_,
        Team_DRtg + 0.2 * (100 * D_Pts_per_ScPoss * (1 - Stop_pct) - Team_DRtg)
      ),

      NetRtg = ORtg - DRtg
    )
}

### Step 12. League reference values (Approach A baseline)

`league_ts_by_season` is the single per-season league true-shooting % used in BPM everywhere in the pipeline.

`league_averages` holds each metric's league mean and SD, shifted forward one season so that each season's Approach-A baseline is the **prior** season's league values.

In [ ]:
league_ts_by_season <- player_full |>
  filter(did_not_play == FALSE, active == TRUE, minutes > 0) |>
  group_by(Season) |>
  summarise(
    lg_ts = sum(points, na.rm = TRUE) /
              (2 * (sum(field_goals_attempted, na.rm = TRUE) +
                    0.44 * sum(free_throws_attempted, na.rm = TRUE))),
    .groups = "drop"
  )

league_stats_raw <- player_full |>
  filter(did_not_play == FALSE, active == TRUE, minutes > 0) |>
  mutate(minutes = as.numeric(minutes)) |>
  left_join(team_gamelog,        by = c("game_id", "Team", "Season")) |>
  left_join(league_ts_by_season, by = "Season") |>
  compute_box_metrics() |>
  compute_oliver_ratings()

league_averages <- league_stats_raw |>
  group_by(Season) |>
  summarise(
    league_avg_PIE        = mean(PIE,        na.rm = TRUE),
    league_sd_PIE         = sd(PIE,          na.rm = TRUE),
    league_avg_Usage_Rate = mean(Usage_Rate, na.rm = TRUE),
    league_sd_Usage_Rate  = sd(Usage_Rate,   na.rm = TRUE),
    league_avg_BPM        = mean(BPM,        na.rm = TRUE),
    league_sd_BPM         = sd(BPM,          na.rm = TRUE),
    league_avg_NetRtg     = mean(NetRtg,     na.rm = TRUE),
    league_sd_NetRtg      = sd(NetRtg,       na.rm = TRUE),
    .groups = "drop"
  ) |>
  mutate(Season = Season + 1)   # season s uses league values from season s - 1

# Coverage check: stints in a season with no prior-season reference
# loaded (the first season in `seasons`) have no Approach-A baseline,
# so the Approach-A chart cannot signal for them.
qualifying_with_next_injury |>
  left_join(league_averages, by = "Season") |>
  summarise(n_stints = n(),
            n_without_league_baseline = sum(is.na(league_avg_PIE)),
            .by = Season) |>
  arrange(Season)

### Step 13. Post-return analysis window
Each qualifying stint's games from the return game through the game before the next injury (or game 82).

In [ ]:
player_gamelog <- player_full |>
  filter(did_not_play == FALSE, active == TRUE, minutes > 0) |>
  mutate(minutes = as.numeric(minutes)) |>
  select(
    athlete_id, athlete_display_name, Season, game_id, game_number, Team,
    minutes,
    points, field_goals_made, field_goals_attempted,
    three_point_field_goals_made,
    free_throws_made, free_throws_attempted,
    offensive_rebounds, defensive_rebounds,
    assists, steals, blocks, fouls, turnovers
  )

game_by_game_raw <- qualifying_with_next_injury |>
  select(athlete_id, Season, Injury_Stint, game_returned, analysis_window_end) |>
  left_join(player_gamelog, by = c("athlete_id", "Season")) |>
  filter(game_number >= game_returned, game_number <= analysis_window_end) |>
  left_join(team_gamelog,        by = c("game_id", "Team", "Season")) |>
  left_join(league_ts_by_season, by = "Season") |>
  compute_box_metrics() |>
  mutate(games_since_return = game_number - game_returned + 1)

### Step 14. Per-game ORtg, DRtg and NetRtg for the post-return window

In [ ]:
game_by_game <- game_by_game_raw |>
  compute_oliver_ratings() |>
  select(
    athlete_id, athlete_display_name, Season, Injury_Stint,
    Team, game_id, game_number, games_since_return,
    minutes,
    PIE, Usage_Rate, BPM,
    ORtg, DRtg, NetRtg
  )

## Part 2 — Control-chart machinery

### Step 15. EWMA-CUSUM statistic

Operates on the lower-sided process capability index

$$\text{PCI}_{\text{lower}} = \frac{x - \text{LSL}}{\sigma_0}, \qquad \text{LSL} = \mu_0 - 2\sigma_0,$$

so $\text{PCI}_{\text{lower}} - 2 = (x - \mu_0)/\sigma_0$. The statistic is $c_j = \min\{0,\ \lambda(\text{PCI}_{\text{lower},j} - 2) + (1-\lambda)c_{j-1}\}$ with $c_0 = 0$, and a signal is raised when $c_j < -h$.

In [ ]:
run_ewma_cusum <- function(values, mu_0, sigma_0, lambda, threshold) {

  n         <- length(values)
  ewma_stat <- numeric(n)
  signal    <- logical(n)
  pci_lower <- numeric(n)

  for (i in seq_len(n)) {
    pci_lower[i] <- (values[i] - (mu_0 - 2 * sigma_0)) / sigma_0

    prev_stat <- if (i == 1) 0 else ewma_stat[i - 1]
    deviation <- pci_lower[i] - 2

    updated      <- lambda * deviation + (1 - lambda) * prev_stat
    ewma_stat[i] <- min(updated, 0)
    signal[i]    <- ewma_stat[i] < -abs(threshold)
  }

  tibble(
    game_index = seq_len(n),
    value      = values,
    pci_lower  = pci_lower,
    ewma_stat  = ewma_stat,
    signal     = signal,
    mu_0       = mu_0,
    sigma_0    = sigma_0
  )
}

### Step 16. (λ, h) parameter grid
Swept during training-season selection in Part 3.

In [ ]:
lambda_grid    <- seq(0.45, 0.95, by = 0.10)
threshold_grid <- seq(0.05, 0.50, by = 0.05)

param_grid <- expand_grid(lambda = lambda_grid, threshold = threshold_grid)

### Step 17. Pre-injury ("healthy game") panel
All games each player played before the first missed game of their qualifying stint.

In [ ]:
metrics_to_model <- c("PIE", "Usage_Rate", "BPM", "NetRtg")

pre_injury_panel <- league_stats_raw |>
  inner_join(
    qualifying_with_next_injury |>
      select(athlete_id, Season, Injury_Stint, first_game_missed),
    by = c("athlete_id", "Season")
  ) |>
  filter(game_number < first_game_missed) |>
  arrange(athlete_id, Season, game_number) |>
  group_by(athlete_id, Injury_Stint) |>
  mutate(pre_game_index = row_number()) |>   # time index for the block bootstrap (Step 23)
  ungroup()

### Step 18. Individualized baseline (empirical-Bayes shrinkage)

Computed per metric from whichever pre-injury panel is supplied (training seasons only in Step 21):

* $\mu_{0,i} = \dfrac{n_i \bar{x}_i + k\,\bar{x}}{n_i + k}$ — a credibility-weighted average of player *i*'s pre-injury mean $\bar{x}_i$ (from $n_i$ games) and the grand mean $\bar{x}$. As $n_i \to \infty$, $\mu_{0,i} \to \bar{x}_i$; as $n_i \to 0$, $\mu_{0,i} \to \bar{x}$.
* $\sigma_0$ — pooled within-player SD (ANOVA $\text{SS}_W / \text{df}_W$).
* $k = \sigma^2_{\text{within}} / \sigma^2_{\text{between}}$, with the between-player variance estimated by the one-way random-effects ANOVA method of moments (Henderson's Method I for unbalanced data). Can be overridden with `k_override`.

The returned table carries the grand mean and pooled SD as attributes (`"grand_mean"`, `"pooled_sd"`), used as the fallback baseline for players without training-season history.

In [ ]:
compute_weighted_baseline <- function(metric_name, data = pre_injury_panel,
                                      k_override = NULL) {

  panel <- data |>
    filter(!is.na(.data[[metric_name]])) |>
    rename(value = all_of(metric_name))

  grand_mean <- mean(panel$value, na.rm = TRUE)
  N          <- nrow(panel)

  player_stats <- panel |>
    group_by(athlete_id) |>
    summarise(
      n_i         = n(),
      player_mean = mean(value, na.rm = TRUE),
      player_var  = if (n() >= 2) var(value, na.rm = TRUE) else NA_real_,
      .groups     = "drop"
    )

  K <- nrow(player_stats)

  # Pooled within-player variance: SSW / df_within
  pooled_var <- player_stats |>
    filter(n_i >= 2) |>
    summarise(pv = sum((n_i - 1) * player_var) / sum(n_i - 1)) |>
    pull(pv)

  if (is.null(k_override)) {
    # Between-player variance by method of moments (unbalanced one-way
    # random effects): (MSB - MSW) / n0, floored at zero.
    SSB <- sum(player_stats$n_i * (player_stats$player_mean - grand_mean)^2)
    MSB <- SSB / (K - 1)
    n0  <- (N - sum(player_stats$n_i^2) / N) / (K - 1)
    between_var <- max((MSB - pooled_var) / n0, 0)

    # k = Inf (full shrinkage to the grand mean) if no between-player variance
    k <- if (between_var > 0) pooled_var / between_var else Inf
  } else {
    k <- k_override
  }

  out <- player_stats |>
    mutate(
      metric  = metric_name,
      mu_0    = (n_i * player_mean + k * grand_mean) / (n_i + k),
      sigma_0 = sqrt(pooled_var),
      k_used  = k
    ) |>
    select(athlete_id, metric, mu_0, sigma_0, n_i, k_used)

  attr(out, "grand_mean") <- grand_mean
  attr(out, "pooled_sd")  <- sqrt(pooled_var)
  out
}

### Step 19. Control-chart runners and confusion-matrix summary

One implementation of "run the chart" per baseline approach, and one confusion-matrix summary, used for the training-season grid search, the held-out evaluation and the ROC sweep.

* `run_chart_metric_league_avg()` — Approach A (prior-season league mean/SD).
* `run_chart_metric_individualized()` — Approach B (`baseline_table` supplies `mu_0` / `sigma_0`).
* `summarise_confusion()` — a stint is predicted positive if the chart signals at any point in its post-return window; optionally stratified by additional grouping columns (e.g. `injury_type`).

In [ ]:
run_chart_metric_league_avg <- function(metric_name, season_filter, grid) {
  avg_col <- paste0("league_avg_", metric_name)
  sd_col  <- paste0("league_sd_",  metric_name)

  qualifying_with_next_injury |>
    filter(Season %in% season_filter) |>
    select(athlete_id, Season, Injury_Stint, had_second_injury,
           injury_type, second_injury_stint) |>
    left_join(league_averages, by = "Season") |>
    left_join(
      game_by_game |>
        filter(Season %in% season_filter) |>
        select(athlete_id, Season, Injury_Stint, game_number,
               games_since_return, athlete_display_name,
               value = all_of(metric_name)),
      by = c("athlete_id", "Season", "Injury_Stint")
    ) |>
    cross_join(grid) |>
    group_by(athlete_id, athlete_display_name, Season, Injury_Stint,
             had_second_injury, injury_type, second_injury_stint,
             lambda, threshold) |>
    arrange(games_since_return, .by_group = TRUE) |>
    reframe(
      run_ewma_cusum(
        values    = value,
        mu_0      = first(.data[[avg_col]]),
        sigma_0   = first(.data[[sd_col]]),
        lambda    = first(lambda),
        threshold = first(threshold)
      ),
      game_number        = game_number,
      games_since_return = games_since_return
    ) |>
    mutate(metric = metric_name)
}

run_chart_metric_individualized <- function(metric_name, baseline_table, season_filter, grid) {
  qualifying_with_next_injury |>
    filter(Season %in% season_filter) |>
    mutate(athlete_id = as.character(athlete_id)) |>
    select(athlete_id, Season, Injury_Stint, had_second_injury,
           injury_type, second_injury_stint) |>
    left_join(
      baseline_table |>
        mutate(athlete_id = as.character(athlete_id)) |>
        filter(metric == metric_name) |>
        select(athlete_id, mu_0, sigma_0, used_league_fallback),
      by = "athlete_id"
    ) |>
    left_join(
      game_by_game |>
        mutate(athlete_id = as.character(athlete_id)) |>
        filter(Season %in% season_filter) |>
        select(athlete_id, Season, Injury_Stint, game_number,
               games_since_return, athlete_display_name,
               value = all_of(metric_name)),
      by = c("athlete_id", "Season", "Injury_Stint")
    ) |>
    cross_join(grid) |>
    group_by(athlete_id, athlete_display_name, Season, Injury_Stint,
             had_second_injury, injury_type, second_injury_stint,
             lambda, threshold) |>
    arrange(games_since_return, .by_group = TRUE) |>
    reframe(
      run_ewma_cusum(
        values    = value,
        mu_0      = first(mu_0),
        sigma_0   = first(sigma_0),
        lambda    = first(lambda),
        threshold = first(threshold)
      ),
      game_number          = game_number,
      games_since_return   = games_since_return,
      value                = value,
      used_league_fallback = first(used_league_fallback)
    ) |>
    mutate(metric = metric_name)
}

summarise_confusion <- function(ewma_df, by = character()) {
  ewma_df |>
    group_by(athlete_id, Season, Injury_Stint, had_second_injury,
             injury_type, metric, lambda, threshold) |>
    summarise(predicted_positive = any(signal, na.rm = TRUE), .groups = "drop") |>
    mutate(
      TP = predicted_positive &  had_second_injury,
      FP = predicted_positive & !had_second_injury,
      TN = !predicted_positive & !had_second_injury,
      FN = !predicted_positive &  had_second_injury
    ) |>
    group_by(across(all_of(by)), metric, lambda, threshold) |>
    summarise(
      TP = sum(TP), FP = sum(FP), TN = sum(TN), FN = sum(FN),
      Sensitivity = TP / (TP + FN),
      Specificity = TN / (TN + FP),
      PPV         = TP / (TP + FP),
      NPV         = TN / (TN + FN),
      Accuracy    = (TP + TN) / (TP + FP + TN + FN),
      .groups     = "drop"
    )
}

# First signal and post-return window length for each true-positive
# (athlete, metric) chart. Because the window ends the game before
# re-injury, the last game_index is the last game before re-injury.
summarise_true_positive_lead_time <- function(ewma_df) {
  ewma_df |>
    filter(had_second_injury == TRUE) |>
    group_by(athlete_id, athlete_display_name, metric, lambda, threshold) |>
    summarise(
      total_games       = max(game_index),
      any_signal        = any(signal, na.rm = TRUE),
      first_signal_game = if (any_signal) min(game_index[signal %in% TRUE]) else NA_real_,
      .groups = "drop"
    ) |>
    filter(any_signal) |>
    mutate(games_signal_to_reinjury = total_games - first_signal_game + 1)
}

## Part 3 — Training/held-out evaluation

### Step 20. Train/test split

The held-out season is `test_season_code`. Seasons are labelled by the calendar year in which they end, so `Season == 2025` is the 2024-25 season; the date-range check below confirms which games are held out.

In [ ]:
test_season_code  <- 2025
test_season_label <- paste0(test_season_code - 1, "-", substr(test_season_code, 3, 4))
train_seasons     <- setdiff(unique(qualifying_with_next_injury$Season), test_season_code)

# Date range of the held-out season
player_full |> filter(Season == test_season_code) |> pull(game_date) |> range()

# Sample size and class balance in each split
qualifying_with_next_injury |>
  mutate(split = if_else(Season == test_season_code, "test", "train")) |>
  count(split, had_second_injury)

### Step 21. Fit individualized baselines — training seasons only

`mu_0`, `sigma_0` and `k` never see a held-out-season game; this frozen baseline is what is deployed on the held-out season. Players with no training-season pre-injury history (e.g. players debuting in the held-out season) fall back to the training-season grand mean and pooled SD.

In [ ]:
pre_injury_panel_train <- pre_injury_panel |>
  filter(Season %in% train_seasons)

baseline_fits_train <- map(metrics_to_model, compute_weighted_baseline,
                           data = pre_injury_panel_train) |>
  set_names(metrics_to_model)

player_baselines_train <- bind_rows(baseline_fits_train)

league_fallback_train <- imap(baseline_fits_train, function(fit, m) {
  tibble(
    metric         = m,
    mu_0_league    = attr(fit, "grand_mean"),
    sigma_0_league = attr(fit, "pooled_sd")
  )
}) |>
  bind_rows()

player_baselines_frozen <- qualifying_with_next_injury |>
  distinct(athlete_id) |>
  cross_join(tibble(metric = metrics_to_model)) |>
  mutate(athlete_id = as.character(athlete_id)) |>
  left_join(
    player_baselines_train |> mutate(athlete_id = as.character(athlete_id)),
    by = c("athlete_id", "metric")
  ) |>
  left_join(league_fallback_train, by = "metric") |>
  mutate(
    used_league_fallback = is.na(mu_0),
    mu_0    = coalesce(mu_0, mu_0_league),
    sigma_0 = coalesce(sigma_0, sigma_0_league)
  ) |>
  select(athlete_id, metric, mu_0, sigma_0, used_league_fallback)

# Number of players using the league fallback baseline, per metric
player_baselines_frozen |> count(metric, used_league_fallback)

### Step 22. Training-season evaluation across the full grid
Every (metric, λ, h) combination for both baseline approaches, training seasons only.

In [ ]:
ewma_train_league_avg <- map_dfr(metrics_to_model, run_chart_metric_league_avg,
                                 season_filter = train_seasons, grid = param_grid)

ewma_train_individualized <- map_dfr(metrics_to_model, function(m)
  run_chart_metric_individualized(m, player_baselines_frozen, train_seasons, param_grid))

confusion_summary_train_league_avg <- summarise_confusion(ewma_train_league_avg) |>
  mutate(approach = "A: league average")

confusion_summary_train_individualized <- summarise_confusion(ewma_train_individualized) |>
  mutate(approach = "B: individualized baseline")

confusion_summary_train_comparison <- bind_rows(
  confusion_summary_train_league_avg,
  confusion_summary_train_individualized
) |>
  select(approach, metric, lambda, threshold, TP, FP, TN, FN,
         Sensitivity, Specificity, PPV, NPV, Accuracy) |>
  arrange(metric, approach, desc(Sensitivity))

confusion_summary_train_comparison

### Step 23. Block-bootstrap null distribution of the EWMA statistic

This is a validation check, not an autocorrelation correction to the control limits:

1. (λ, h) are selected by outcome-based evaluation against observed re-injury (Step 25), not from an analytic in-control variance formula, so threshold selection does not rest on an i.i.d. assumption.
2. A single pooled autocorrelation model (e.g. AR(1)) would impose a common dependence structure across players, while each player's dependence structure is plausibly distinct and non-stationary.

Instead, each player's standardized pre-injury games (training seasons only) are resampled in overlapping blocks, which preserves short-range within-player dependence. For each (metric, λ, h) we estimate how often a synthetic "healthy" sequence would trigger a signal by chance. If the observed false-positive rate among non-re-injured training players is at or below this rate, no further correction is needed.

In [ ]:
pre_injury_panel_std_train <- pre_injury_panel_train |>
  select(athlete_id, Injury_Stint, pre_game_index, all_of(metrics_to_model)) |>
  pivot_longer(all_of(metrics_to_model), names_to = "metric", values_to = "value") |>
  mutate(athlete_id = as.character(athlete_id)) |>
  left_join(player_baselines_frozen, by = c("athlete_id", "metric")) |>
  filter(!is.na(value), !is.na(mu_0)) |>
  mutate(pci_lower = (value - (mu_0 - 2 * sigma_0)) / sigma_0) |>
  arrange(athlete_id, metric, pre_game_index)

ewma_on_pci <- function(pci, lambda) {
  n    <- length(pci)
  stat <- numeric(n)
  for (i in seq_len(n)) {
    dev     <- pci[i] - 2
    prev    <- if (i == 1) 0 else stat[i - 1]
    stat[i] <- min(lambda * dev + (1 - lambda) * prev, 0)
  }
  stat
}

build_blocks <- function(df, block_len) {
  n <- nrow(df)
  if (n < block_len) return(list())
  map(seq_len(n - block_len + 1), ~ df$pci_lower[.x:(.x + block_len - 1)])
}

# Fraction of block-resampled synthetic "healthy" sequences that would
# signal by chance for a given (metric, lambda, threshold)
block_bootstrap_false_alarm_rate <- function(metric_name, lambda, threshold,
                                             block_len = 4, n_boot = 2000,
                                             synth_len = 20) {

  blocks <- pre_injury_panel_std_train |>
    filter(metric == metric_name) |>
    group_by(athlete_id, Injury_Stint) |>
    group_split() |>
    map(build_blocks, block_len = block_len) |>
    list_flatten()

  n_needed <- ceiling(synth_len / block_len)

  false_alarms <- map_lgl(seq_len(n_boot), function(b) {
    seq_vals <- sample(blocks, n_needed, replace = TRUE) |> unlist()
    min(ewma_on_pci(seq_vals, lambda)) < -abs(threshold)
  })

  mean(false_alarms)
}

# Same grid as Step 22: 4 metrics x 6 lambdas x 10 thresholds = 240
# combinations x n_boot = 2000 resamples (slow; reduce n_boot for a quick pass)
set.seed(bootstrap_seed)

bootstrap_null_false_alarm <- expand_grid(
    metric = metrics_to_model, lambda = lambda_grid, threshold = threshold_grid
  ) |>
  mutate(
    bootstrap_null_false_alarm_rate = pmap_dbl(
      list(metric, lambda, threshold),
      block_bootstrap_false_alarm_rate
    )
  )

### Step 24. Autocorrelation validation table (training seasons)

For each (metric, λ, h):

* `observed_false_positive_rate` — empirical false-positive rate among training-season players who were **not** re-injured (Step 22, Approach B);
* `bootstrap_null_false_alarm_rate` — chance signal rate under dependence-preserving resampling (Step 23).

`flag_inflated = TRUE` would indicate the chart signals on real post-return data more often than autocorrelation-preserving chance alone predicts.

In [ ]:
autocorrelation_validation <- confusion_summary_train_individualized |>
  select(metric, lambda, threshold, FP, TN, Specificity) |>
  mutate(observed_false_positive_rate = 1 - Specificity) |>
  left_join(bootstrap_null_false_alarm, by = c("metric", "lambda", "threshold")) |>
  mutate(
    flag_inflated = observed_false_positive_rate > bootstrap_null_false_alarm_rate
  ) |>
  select(metric, lambda, threshold, observed_false_positive_rate,
         bootstrap_null_false_alarm_rate, flag_inflated) |>
  arrange(metric, lambda, threshold)

autocorrelation_validation |> count(metric, flag_inflated)

In [ ]:
autocorrelation_validation_summary <- autocorrelation_validation |>
  group_by(metric) |>
  summarise(
    mean_observed_fp_rate    = mean(observed_false_positive_rate),
    mean_bootstrap_null_rate = mean(bootstrap_null_false_alarm_rate),
    mean_gap                 = mean(bootstrap_null_false_alarm_rate - observed_false_positive_rate),
    max_gap                  = max(bootstrap_null_false_alarm_rate - observed_false_positive_rate),
    min_gap                  = min(bootstrap_null_false_alarm_rate - observed_false_positive_rate),
    n_combinations           = n(),
    .groups = "drop"
  )

autocorrelation_validation_summary

### Step 25. Select (λ, h) per metric — training seasons only, both approaches
Selection rule: highest training-season accuracy on the Step 22 grid.

In [ ]:
selected_params_league_avg <- confusion_summary_train_league_avg |>
  group_by(metric) |> slice_max(Accuracy, n = 1, with_ties = FALSE) |> ungroup() |>
  select(metric, lambda, threshold)

selected_params_individualized <- confusion_summary_train_individualized |>
  group_by(metric) |> slice_max(Accuracy, n = 1, with_ties = FALSE) |> ungroup() |>
  select(metric, lambda, threshold)

selected_params_league_avg
selected_params_individualized

### Step 26. Apply the frozen model and selected parameters to the held-out season
This is the first and only time the held-out season is used; `confusion_summary_test_comparison` gives the headline results.

In [ ]:
ewma_test_league_avg <- map_dfr(metrics_to_model, function(m) {
  p <- selected_params_league_avg |> filter(metric == m) |> select(lambda, threshold)
  run_chart_metric_league_avg(m, season_filter = test_season_code, grid = p)
})

ewma_test_individualized <- map_dfr(metrics_to_model, function(m) {
  p <- selected_params_individualized |> filter(metric == m) |> select(lambda, threshold)
  run_chart_metric_individualized(m, player_baselines_frozen, test_season_code, p)
})

confusion_summary_test_league_avg <- summarise_confusion(ewma_test_league_avg) |>
  mutate(approach = "A: league average -- HELD-OUT SEASON")

confusion_summary_test_individualized <- summarise_confusion(ewma_test_individualized) |>
  mutate(approach = "B: individualized baseline -- HELD-OUT SEASON")

confusion_summary_test_comparison <- bind_rows(
  confusion_summary_test_league_avg,
  confusion_summary_test_individualized
) |>
  select(approach, metric, lambda, threshold, TP, FP, TN, FN,
         Sensitivity, Specificity, PPV, NPV, Accuracy) |>
  arrange(metric, approach)

confusion_summary_test_comparison

## Part 4 — ROC/AUC comparison (training seasons)

### Step 27. Wide threshold grid
The threshold grid is extended well beyond the selection grid so both approaches' specificity can approach 1. The same grid is used for both approaches.

In [ ]:
threshold_grid_wide <- c(seq(0.05, 0.50, by = 0.05), seq(0.75, 5.00, by = 0.25))
param_grid_wide     <- expand_grid(lambda = lambda_grid, threshold = threshold_grid_wide)

ewma_train_A_wide <- map_dfr(metrics_to_model, run_chart_metric_league_avg,
                             season_filter = train_seasons, grid = param_grid_wide)
ewma_train_B_wide <- map_dfr(metrics_to_model, function(m)
  run_chart_metric_individualized(m, player_baselines_frozen, train_seasons, param_grid_wide))

confusion_summary_train_A_wide <- summarise_confusion(ewma_train_A_wide)
confusion_summary_train_B_wide <- summarise_confusion(ewma_train_B_wide)

# Sensitivity vs. (1 - specificity) across the wide grid, per approach
roc_style_comparison <- bind_rows(
  confusion_summary_train_A_wide |> mutate(approach = "A"),
  confusion_summary_train_B_wide |> mutate(approach = "B")
) |>
  mutate(false_positive_rate = 1 - Specificity) |>
  select(metric, approach, lambda, threshold, Sensitivity, false_positive_rate, Specificity) |>
  arrange(metric, approach, lambda, threshold)

roc_style_comparison |>
  filter(metric == "BPM") |>
  arrange(approach, false_positive_rate) |>
  print(n = Inf)

### Step 28. ROC frontier and AUC
The ROC curve for each metric/approach is the Pareto frontier of the (λ, h) grid, anchored at (0, 0) and (1, 1); AUC is computed by trapezoidal integration over the frontier.

In [ ]:
compute_roc_frontier <- function(df) {
  df |>
    group_by(metric, approach) |>
    arrange(false_positive_rate, desc(Sensitivity), .by_group = TRUE) |>
    mutate(running_max_sens = cummax(Sensitivity)) |>
    filter(Sensitivity == running_max_sens) |>
    distinct(false_positive_rate, .keep_all = TRUE) |>
    ungroup() |>
    select(metric, approach, lambda, threshold,
           false_positive_rate, Sensitivity)
}

add_roc_anchors <- function(df) {
  anchors <- df |>
    distinct(metric, approach) |>
    crossing(false_positive_rate = c(0, 1)) |>
    mutate(Sensitivity = false_positive_rate, lambda = NA, threshold = NA)

  bind_rows(df, anchors) |>
    distinct(metric, approach, false_positive_rate, Sensitivity, .keep_all = TRUE) |>
    arrange(metric, approach, false_positive_rate)
}

compute_auc <- function(df) {
  df <- df |> arrange(false_positive_rate)
  sum(diff(df$false_positive_rate) *
        (head(df$Sensitivity, -1) + tail(df$Sensitivity, -1)) / 2)
}

roc_curve_data <- roc_style_comparison |>
  compute_roc_frontier() |>
  add_roc_anchors()

auc_summary <- roc_curve_data |>
  group_by(metric, approach) |>
  summarise(AUC = compute_auc(pick(false_positive_rate, Sensitivity)), .groups = "drop") |>
  pivot_wider(names_from = approach, values_from = AUC, names_prefix = "AUC_") |>
  mutate(AUC_diff = AUC_B - AUC_A)

auc_summary

## Part 5 — Figures

### Step 29. Parameter-grid heatmaps (Approach B, training seasons)
Greyscale 2×2 grids of sensitivity and specificity across (λ, h), one panel per metric.

In [ ]:
metric_labels <- c(
  "PIE"        = "PIE",
  "Usage_Rate" = "Usage Rate",
  "BPM"        = "BPM",
  "NetRtg"     = "Net Rating"
)

panel_tags <- c("A", "B", "C", "D")

# One heatmap panel; optionally restricted to a single injury area
make_heatmap_grey <- function(metric_name, perf_metric, data, inj_type = NULL,
                              fill_low = "grey95", fill_high = "grey20") {

  df <- data |> filter(metric == .env$metric_name)
  if (!is.null(inj_type)) df <- df |> filter(injury_type == .env$inj_type)

  # Switch label colour so text stays legible on light and dark tiles
  df <- df |> mutate(text_color = if_else(.data[[perf_metric]] > 0.55, "white", "black"))

  ggplot(df, aes(x = threshold, y = lambda, fill = .data[[perf_metric]])) +
    geom_tile(color = "white", linewidth = 0.8) +
    geom_text(
      aes(label = sprintf("%.2f", .data[[perf_metric]]), color = text_color),
      fontface = "bold", size = 3.3
    ) +
    scale_color_identity() +
    scale_fill_gradient(
      low      = fill_low,
      high     = fill_high,
      limits   = c(0, 1),
      na.value = "grey80",
      name     = perf_metric,
      labels   = scales::percent_format(accuracy = 1)
    ) +
    labs(
      title = metric_labels[metric_name],
      x     = "Threshold (h)",
      y     = "Smoothing Weight (λ)"
    ) +
    theme_minimal(base_size = 11) +
    theme(
      plot.title      = element_text(face = "bold", hjust = 0.5),
      axis.text.x     = element_text(color = "grey20"),
      axis.text.y     = element_text(color = "grey20"),
      panel.grid      = element_blank(),
      legend.position = "right"
    )
}

# 2x2 patchwork grid (one panel per metric) for one performance measure
make_grid_grey <- function(perf_metric, data, title, subtitle = "Training Data",
                           inj_type = NULL) {

  plots <- imap(metric_labels, function(label, m) {
    make_heatmap_grey(m, perf_metric, data, inj_type = inj_type)
  })

  (plots[[1]] | plots[[2]]) /
  (plots[[3]] | plots[[4]]) +
    plot_annotation(
      title      = title,
      subtitle   = subtitle,
      tag_levels = list(panel_tags)
    ) &
    theme(plot.tag = element_text(face = "bold", size = 12))
}

grid_sensitivity_grey <- make_grid_grey(
  "Sensitivity", confusion_summary_train_individualized,
  title = "EWMA-CUSUM Parameter Grid — Sensitivity"
)

grid_specificity_grey <- make_grid_grey(
  "Specificity", confusion_summary_train_individualized,
  title = "EWMA-CUSUM Parameter Grid — Specificity"
)

ggsave("heatmap_overall_sensitivity_greyscale.png", grid_sensitivity_grey,
       width = 12, height = 9, dpi = 300, bg = "white")
ggsave("heatmap_overall_specificity_greyscale.png", grid_specificity_grey,
       width = 12, height = 9, dpi = 300, bg = "white")

print(grid_sensitivity_grey)
print(grid_specificity_grey)

### Step 30. Heatmaps stratified by injury area
Same confusion-matrix logic, keeping `injury_type` in the final grouping. Figures are produced for upper-extremity, lower-extremity and core injuries.

In [ ]:
confusion_summary_train_individualized_by_type <-
  summarise_confusion(ewma_train_individualized, by = "injury_type") |>
  filter(injury_type %in% c("Upper Extremity", "Lower Extremity", "Core", "Other"))

injury_areas <- c("Upper Extremity", "Lower Extremity", "Core")

injury_type_grids <- map(injury_areas, function(inj_type) {

  grid_sensitivity <- make_grid_grey(
    "Sensitivity", confusion_summary_train_individualized_by_type,
    title = paste0(inj_type, " — Sensitivity"), inj_type = inj_type
  )
  grid_specificity <- make_grid_grey(
    "Specificity", confusion_summary_train_individualized_by_type,
    title = paste0(inj_type, " — Specificity"), inj_type = inj_type
  )

  inj_slug <- str_replace_all(str_to_lower(inj_type), " ", "_")

  ggsave(paste0("heatmap_", inj_slug, "_sensitivity_greyscale.png"),
         grid_sensitivity, width = 12, height = 9, dpi = 300, bg = "white")
  ggsave(paste0("heatmap_", inj_slug, "_specificity_greyscale.png"),
         grid_specificity, width = 12, height = 9, dpi = 300, bg = "white")

  list(sensitivity = grid_sensitivity, specificity = grid_specificity)
}) |>
  set_names(injury_areas)

# Six figures: 3 injury areas x 2 performance measures
walk(injury_type_grids, function(pair) {
  print(pair$sensitivity)
  print(pair$specificity)
})

### Step 31. Example control chart (held-out season)

Candidate true positives (signal fired and player was re-injured) under each metric's selected (λ, h), ranked by how close the lead time from first signal to re-injury is to 10 games. The example shown is Deni Avdija (`athlete_id` 4683021), Usage Rate.

In [ ]:
selected_grid_by_metric <- selected_params_individualized |>
  split(~ metric)

ewma_test_selected <- map_dfr(names(selected_grid_by_metric), function(m) {
  run_chart_metric_individualized(
    m, player_baselines_frozen, test_season_code,
    grid = selected_grid_by_metric[[m]] |> select(lambda, threshold)
  )
})

candidate_true_positives <- summarise_true_positive_lead_time(ewma_test_selected) |>
  mutate(gap_from_target = abs(games_signal_to_reinjury - 10)) |>
  arrange(gap_from_target, desc(total_games))

candidate_true_positives |> print(n = 15)

In [ ]:
chosen_athlete_id <- "4683021"   # Deni Avdija
chosen_metric     <- "Usage_Rate"

chart_data <- ewma_test_selected |>
  filter(athlete_id == chosen_athlete_id, metric == chosen_metric)

chosen_h         <- selected_grid_by_metric[[chosen_metric]]$threshold[1]
first_signal_idx <- chart_data |> filter(signal) |> slice(1)

example_chart <- ggplot(chart_data, aes(x = game_index, y = ewma_stat)) +
  geom_hline(yintercept = -abs(chosen_h), linetype = "dashed", color = "grey40") +
  geom_line(color = "grey20", linewidth = 0.7) +
  geom_point(aes(shape = signal), size = 2.4, color = "grey20") +
  geom_point(data = first_signal_idx, aes(x = game_index, y = ewma_stat),
             shape = 21, size = 4.5, fill = "grey40", color = "black", stroke = 1) +
  annotate("text", x = max(chart_data$game_index), y = -abs(chosen_h),
           label = paste0("h = ", -abs(chosen_h)), vjust = -0.6, hjust = 1,
           size = 3.3, color = "grey40") +
  scale_shape_manual(values = c(`TRUE` = 17, `FALSE` = 16), guide = "none") +
  scale_x_continuous(breaks = seq(1, max(chart_data$game_index), by = 1)) +
  labs(
    title    = paste0("Monitoring the ", metric_labels[chosen_metric]),
    subtitle = paste0("Example EWMA control chart, ", test_season_label, " season"),
    x = "Games Since Return", y = expression(c[ij])
  ) +
  theme_minimal(base_size = 12) +
  theme(plot.title = element_text(face = "bold"), panel.grid.minor = element_blank())

ggsave("example_control_chart_avdija_usage_rate.png", example_chart,
       width = 8, height = 5, dpi = 300, bg = "white")

example_chart

## Part 6 — Descriptive and supplementary statistics

### Step 32. Games missed in qualifying stints (mean and 95% CI)
One row per athlete-season (the first qualifying stint), i.e. the analysis unit used throughout.

In [ ]:
games_missed_ci <- t.test(qualifying_with_next_injury$actual_games_missed)

games_missed_summary <- tibble(
  n        = nrow(qualifying_with_next_injury),
  mean     = unname(games_missed_ci$estimate),
  ci_lower = games_missed_ci$conf.int[1],
  ci_upper = games_missed_ci$conf.int[2],
  sd       = sd(qualifying_with_next_injury$actual_games_missed)
)

games_missed_summary

### Step 33. Lead time from first signal to re-injury (true positives)
Held-out season, Approach B, selected (λ, h).

In [ ]:
tp_lead_time <- summarise_true_positive_lead_time(ewma_test_individualized)

tp_lead_time

tp_lead_time_summary <- tp_lead_time |>
  summarise(
    n      = n(),
    mean   = mean(games_signal_to_reinjury),
    sd     = sd(games_signal_to_reinjury),
    median = median(games_signal_to_reinjury),
    min    = min(games_signal_to_reinjury),
    max    = max(games_signal_to_reinjury),
    q1     = unname(quantile(games_signal_to_reinjury, 0.25)),
    q3     = unname(quantile(games_signal_to_reinjury, 0.75))
  )

tp_lead_time_summary

tp_lead_time_by_metric <- tp_lead_time |>
  group_by(metric) |>
  summarise(
    n      = n(),
    mean   = mean(games_signal_to_reinjury),
    sd     = sd(games_signal_to_reinjury),
    median = median(games_signal_to_reinjury),
    .groups = "drop"
  )

tp_lead_time_by_metric

### Step 34. Wald 95% CIs for sensitivity, specificity and accuracy at the selected (λ, h)
Approach B, training seasons.

In [ ]:
wald_ci <- function(p, n) 1.96 * sqrt(p * (1 - p) / n)

confusion_summary_train_individualized |>
  group_by(metric) |> slice_max(Accuracy, n = 1, with_ties = FALSE) |> ungroup() |>
  mutate(
    Sens_N = TP + FN,
    Spec_N = TN + FP,
    N      = TP + FP + TN + FN,
    Sens_U = Sensitivity + wald_ci(Sensitivity, Sens_N),
    Sens_L = Sensitivity - wald_ci(Sensitivity, Sens_N),
    Spec_U = Specificity + wald_ci(Specificity, Spec_N),
    Spec_L = Specificity - wald_ci(Specificity, Spec_N),
    Accu_U = Accuracy    + wald_ci(Accuracy,    N),
    Accu_L = Accuracy    - wald_ci(Accuracy,    N)
  ) |>
  select(metric, lambda, threshold, Sensitivity, Sens_U, Sens_L,
         Specificity, Spec_U, Spec_L, Accuracy, Accu_U, Accu_L)

### Step 35. Highest-accuracy point on the wide grid (Approach B, training seasons)
On the wide grid, the highest-accuracy settings are near-trivial classifiers (specificity ≈ 1, sensitivity ≈ 0), which is why (λ, h) selection uses the narrower Step 16 grid and why the ROC/AUC comparison is reported.

In [ ]:
confusion_summary_train_B_wide |>
  group_by(metric) |>
  slice_max(Accuracy, n = 1, with_ties = FALSE) |>
  ungroup()

### Step 36. Games from return to re-injury (all re-injured players)
Independent of chart signal, metric, (λ, h) and train/test split.

In [ ]:
games_until_reinjury <- qualifying_with_next_injury |>
  filter(had_second_injury == TRUE) |>
  left_join(
    game_by_game |>
      group_by(athlete_id, Season, Injury_Stint) |>
      summarise(games_until_reinjury = max(games_since_return), .groups = "drop"),
    by = c("athlete_id", "Season", "Injury_Stint")
  )

games_until_reinjury_summary <- games_until_reinjury |>
  summarise(
    n      = sum(!is.na(games_until_reinjury)),
    mean   = mean(games_until_reinjury, na.rm = TRUE),
    sd     = sd(games_until_reinjury, na.rm = TRUE),
    median = median(games_until_reinjury, na.rm = TRUE),
    min    = min(games_until_reinjury, na.rm = TRUE),
    max    = max(games_until_reinjury, na.rm = TRUE),
    q1     = unname(quantile(games_until_reinjury, 0.25, na.rm = TRUE)),
    q3     = unname(quantile(games_until_reinjury, 0.75, na.rm = TRUE))
  ) |>
  mutate(
    CIL = mean - qt(0.975, df = n - 1) * sd / sqrt(n),
    CIU = mean + qt(0.975, df = n - 1) * sd / sqrt(n)
  )

games_until_reinjury_summary

### Step 37. Players with a qualifying stint, by season

In [ ]:
qualifying_with_next_injury |>
  distinct(Season, athlete_id) |>
  count(Season)

### Step 38. Post-return mean metric values: re-injured vs. not re-injured
Per-stint mean of each metric over the post-return window, compared with Welch two-sample t-tests.

In [ ]:
player_mean_metrics <- game_by_game |>
  left_join(
    qualifying_with_next_injury |>
      select(athlete_id, Season, Injury_Stint, had_second_injury),
    by = c("athlete_id", "Season", "Injury_Stint")
  ) |>
  group_by(athlete_id, athlete_display_name, Season, Injury_Stint, had_second_injury) |>
  summarise(
    across(all_of(metrics_to_model), \(x) mean(x, na.rm = TRUE)),
    .groups = "drop"
  )

player_mean_metrics |>
  group_by(had_second_injury) |>
  summarise(
    Mean_PIE = mean(PIE,        na.rm = TRUE),
    Mean_UR  = mean(Usage_Rate, na.rm = TRUE),
    Mean_BP  = mean(BPM,        na.rm = TRUE),
    Mean_NR  = mean(NetRtg,     na.rm = TRUE),
    n        = n()
  )

metric_ttests <- map_dfr(metrics_to_model, function(m) {
  result <- t.test(player_mean_metrics[[m]] ~ player_mean_metrics$had_second_injury)

  tibble(
    metric         = m,
    mean_not_reinj = result$estimate[1],
    mean_reinj     = result$estimate[2],
    mean_diff      = result$estimate[1] - result$estimate[2],
    ci_lower       = result$conf.int[1],
    ci_upper       = result$conf.int[2],
    t_stat         = result$statistic,
    df             = result$parameter,
    p_value        = result$p.value
  )
})

metric_ttests

### Step 39. Pre- vs. post-injury panel
One row per stint with the player's mean pre-injury value (`<metric>_pre`, Step 17 panel), mean post-return value (`<metric>_post`, Step 38) and change (`<metric>_diff = post - pre`). Stints with no pre-injury games in that season are excluded.

In [ ]:
pre_injury_means <- pre_injury_panel |>
  group_by(athlete_id, Season, Injury_Stint) |>
  summarise(
    across(all_of(metrics_to_model), \(x) mean(x, na.rm = TRUE), .names = "{.col}_pre"),
    .groups = "drop"
  )

player_pre_post <- player_mean_metrics |>
  rename_with(\(x) paste0(x, "_post"), all_of(metrics_to_model)) |>
  inner_join(pre_injury_means, by = c("athlete_id", "Season", "Injury_Stint"))

for (m in metrics_to_model) {
  player_pre_post[[paste0(m, "_diff")]] <-
    player_pre_post[[paste0(m, "_post")]] - player_pre_post[[paste0(m, "_pre")]]
}

player_pre_post |> count(had_second_injury)

### Step 40. Difference-in-differences: Welch t-tests and Cohen's *d*
Compares each stint's (post − pre) change between re-injured and not re-injured players; *d* uses the pooled SD of the change scores.

In [ ]:
did_ttests <- map_dfr(metrics_to_model, function(m) {
  diff_col <- paste0(m, "_diff")

  group0 <- player_pre_post[[diff_col]][player_pre_post$had_second_injury == FALSE]
  group1 <- player_pre_post[[diff_col]][player_pre_post$had_second_injury == TRUE]

  result <- t.test(player_pre_post[[diff_col]] ~ player_pre_post$had_second_injury)

  n0  <- sum(!is.na(group0)); n1 <- sum(!is.na(group1))
  sd0 <- sd(group0, na.rm = TRUE); sd1 <- sd(group1, na.rm = TRUE)
  pooled_sd <- sqrt(((n0 - 1) * sd0^2 + (n1 - 1) * sd1^2) / (n0 + n1 - 2))
  cohens_d  <- (mean(group0, na.rm = TRUE) - mean(group1, na.rm = TRUE)) / pooled_sd

  tibble(
    metric              = m,
    n_not_reinj         = n0,
    n_reinj             = n1,
    mean_diff_not_reinj = result$estimate[1],
    mean_diff_reinj     = result$estimate[2],
    did_estimate        = result$estimate[1] - result$estimate[2],
    ci_lower            = result$conf.int[1],
    ci_upper            = result$conf.int[2],
    t_stat              = result$statistic,
    df                  = result$parameter,
    p_value             = result$p.value,
    cohens_d            = cohens_d
  )
})

did_ttests

### Step 41. Nonparametric tests
**Wilcoxon rank-sum (Mann–Whitney):** the (post − pre) change compared between independent groups (re-injured vs. not), with the Hodges–Lehmann estimate of the location shift.

**Wilcoxon signed-rank (paired, within group):** post- vs. pre-injury values for the same players, separately for each group.

In [ ]:
did_wilcox <- map_dfr(metrics_to_model, function(m) {
  diff_col <- paste0(m, "_diff")
  result <- wilcox.test(
    player_pre_post[[diff_col]] ~ player_pre_post$had_second_injury,
    conf.int = TRUE
  )

  tibble(
    metric                = m,
    median_diff_not_reinj = median(player_pre_post[[diff_col]][player_pre_post$had_second_injury == FALSE], na.rm = TRUE),
    median_diff_reinj     = median(player_pre_post[[diff_col]][player_pre_post$had_second_injury == TRUE],  na.rm = TRUE),
    hl_estimate           = result$estimate,   # Hodges-Lehmann location shift
    ci_lower              = result$conf.int[1],
    ci_upper              = result$conf.int[2],
    W_stat                = result$statistic,
    p_value               = result$p.value
  )
})

did_wilcox

In [ ]:
signed_rank_within_group <- map_dfr(metrics_to_model, function(m) {
  pre_col  <- paste0(m, "_pre")
  post_col <- paste0(m, "_post")

  map_dfr(c(FALSE, TRUE), function(status) {
    sub    <- player_pre_post |> filter(had_second_injury == status)
    result <- wilcox.test(sub[[post_col]], sub[[pre_col]], paired = TRUE, conf.int = TRUE)

    tibble(
      metric            = m,
      had_second_injury = status,
      n                 = nrow(sub),
      median_pre        = median(sub[[pre_col]],  na.rm = TRUE),
      median_post       = median(sub[[post_col]], na.rm = TRUE),
      hl_estimate       = result$estimate,
      ci_lower          = result$conf.int[1],
      ci_upper          = result$conf.int[2],
      V_stat            = result$statistic,
      p_value           = result$p.value
    )
  })
})

signed_rank_within_group

## Optional: export result tables
Set `export_tables <- TRUE` to write the main result tables to an Excel workbook (requires the `writexl` package).

In [ ]:
export_tables <- FALSE

if (export_tables) {
  if (!requireNamespace("writexl", quietly = TRUE)) install.packages("writexl")
  writexl::write_xlsx(
    list(
      "train_full_grid"             = confusion_summary_train_comparison,
      "autocorr_validation"         = autocorrelation_validation,
      "autocorr_validation_summary" = autocorrelation_validation_summary,
      "test_holdout_comparison"     = confusion_summary_test_comparison,
      "auc_summary"                 = auc_summary
    ),
    "reinjury_ewma_confusion_matrices.xlsx"
  )
}

## Session information

In [ ]:
sessionInfo()